In [1]:
py_path <- Sys.which("python")

if (py_path == "") {
  py_path <- Sys.which("python3")
}

Sys.setenv(RETICULATE_PYTHON = py_path)

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(reticulate)
  library(CellEnergy)
})

cat("[INFO] Python used by reticulate:\n")
print(reticulate::py_config())

cat("[INFO] Python module check:\n")
cat("pandas:", reticulate::py_module_available("pandas"), "\n")
cat("numpy :", reticulate::py_module_available("numpy"), "\n")
cat("scipy :", reticulate::py_module_available("scipy"), "\n\n")

Warning message:
“no DISPLAY variable so Tk is not available”


[INFO] Python used by reticulate:
python:         /home/zhanghang/Anaconda/envs/r_cellenergy/bin/python
libpython:      /home/zhanghang/Anaconda/envs/r_cellenergy/lib/libpython3.14.so
pythonhome:     /home/zhanghang/Anaconda/envs/r_cellenergy:/home/zhanghang/Anaconda/envs/r_cellenergy
version:        3.14.5 | packaged by conda-forge | (main, May 20 2026, 00:15:36) [GCC 14.3.0]
numpy:          /home/zhanghang/Anaconda/envs/r_cellenergy/lib/python3.14/site-packages/numpy
numpy_version:  2.4.6

NOTE: Python version was forced by RETICULATE_PYTHON
[INFO] Python module check:
pandas: TRUE 
numpy : TRUE 
scipy : TRUE 



In [ ]:

py_path <- Sys.which("python")

if (py_path == "") {
  py_path <- Sys.which("python3")
}

Sys.setenv(RETICULATE_PYTHON = py_path)

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(reticulate)
  library(CellEnergy)
})

cat("[INFO] Python used by reticulate:\n")
print(reticulate::py_config())

cat("[INFO] Python module check:\n")
cat("pandas:", reticulate::py_module_available("pandas"), "\n")
cat("numpy :", reticulate::py_module_available("numpy"), "\n")
cat("scipy :", reticulate::py_module_available("scipy"), "\n\n")



data_dir <- "path/GSE132188/mm10"

mtx_path <- file.path(data_dir, "matrix.mtx")
gene_path <- file.path(data_dir, "genes.tsv")
barcode_path <- file.path(data_dir, "barcodes.tsv")
barcode_meta_path <- file.path(data_dir, "barcode.csv")

out_dir <- "path/Seurat_CellEnergy_full_pipeline"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)



min_cells <- 3
min_features <- 200
n_hvg <- 2000
scale_factor <- 10000

label_col <- "clusters_fig6_broad_final"

cat("[INFO] Input directory:\n")
cat(data_dir, "\n\n")

cat("[INFO] Output directory:\n")
cat(out_dir, "\n\n")

if (!file.exists(mtx_path)) {
  stop("[ERROR] matrix.mtx not found: ", mtx_path)
}

if (!file.exists(gene_path)) {
  stop("[ERROR] genes.tsv not found: ", gene_path)
}

if (!file.exists(barcode_path)) {
  stop("[ERROR] barcodes.tsv not found: ", barcode_path)
}

if (!file.exists(barcode_meta_path)) {
  stop("[ERROR] barcode.csv not found: ", barcode_meta_path)
}


write_matrix_with_id <- function(mat, file, id_col = "ID") {
  mat <- as.matrix(mat)
  df <- as.data.frame(mat, check.names = FALSE)
  df <- cbind(setNames(data.frame(rownames(df), check.names = FALSE), id_col), df)
  data.table::fwrite(df, file = file)
}

minmax_scale_columns <- function(mat) {
  mat <- as.matrix(mat)

  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)
  col_range <- col_max - col_min

  col_range[col_range == 0 | is.na(col_range)] <- 1

  scaled <- sweep(mat, 2, col_min, FUN = "-")
  scaled <- sweep(scaled, 2, col_range, FUN = "/")
  scaled[is.na(scaled)] <- 0

  return(scaled)
}

get_assay_data_compat <- function(obj, assay = "RNA", layer_name = "counts", slot_name = "counts") {
  DefaultAssay(obj) <- assay

  tryCatch(
    GetAssayData(obj, assay = assay, layer = layer_name),
    error = function(e) {
      GetAssayData(obj, assay = assay, slot = slot_name)
    }
  )
}



cat("[INFO] Reading matrix.mtx / genes.tsv / barcodes.tsv...\n")

counts <- ReadMtx(
  mtx = mtx_path,
  features = gene_path,
  cells = barcode_path,
  feature.column = 2,
  cell.column = 1,
  unique.features = TRUE
)

cat("[INFO] Raw matrix dimension, genes x cells:\n")
print(dim(counts))

saveRDS(
  counts,
  file.path(out_dir, "GSE132188_raw_counts_gene_by_cell.rds")
)



seurat_obj <- CreateSeuratObject(
  counts = counts,
  project = "GSE132188",
  min.cells = min_cells,
  min.features = min_features
)

cat("[INFO] Seurat object after min.cells / min.features filtering:\n")
print(seurat_obj)

qc_summary_before_label <- data.frame(
  n_genes_before_qc = nrow(counts),
  n_cells_before_qc = ncol(counts),
  n_genes_after_basic_qc = nrow(seurat_obj),
  n_cells_after_basic_qc = ncol(seurat_obj),
  min_cells = min_cells,
  min_features = min_features,
  n_hvg = n_hvg,
  scale_factor = scale_factor
)

data.table::fwrite(
  qc_summary_before_label,
  file.path(out_dir, "QC_summary_before_label_filter.csv")
)



cat("[INFO] Reading annotation file barcode.csv...\n")

barcode_meta <- data.table::fread(
  barcode_meta_path,
  header = TRUE,
  data.table = FALSE
)

cat("[INFO] barcode metadata dimension:\n")
print(dim(barcode_meta))

cat("[INFO] barcode metadata columns:\n")
print(colnames(barcode_meta))


colnames(barcode_meta)[1] <- "Cell"

for (cn in colnames(barcode_meta)) {
  barcode_meta[[cn]] <- gsub('"', '', as.character(barcode_meta[[cn]]))
}

if (!label_col %in% colnames(barcode_meta)) {
  stop("[ERROR] label_col not found in barcode.csv: ", label_col)
}

barcode_meta <- barcode_meta[
  !is.na(barcode_meta[[label_col]]) &
    barcode_meta[[label_col]] != "" &
    barcode_meta[[label_col]] != "Excluded",
  ,
  drop = FALSE
]

rownames(barcode_meta) <- barcode_meta$Cell

cat("[INFO] Label distribution before alignment:\n")
print(table(barcode_meta[[label_col]], useNA = "ifany"))

common_cells <- intersect(colnames(seurat_obj), rownames(barcode_meta))

cat("[INFO] Common cells between Seurat object and barcode.csv:\n")
print(length(common_cells))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cells between Seurat object and barcode.csv")
}

seurat_obj <- subset(seurat_obj, cells = common_cells)

barcode_meta <- barcode_meta[colnames(seurat_obj), , drop = FALSE]

seurat_obj <- AddMetaData(
  object = seurat_obj,
  metadata = barcode_meta
)


seurat_obj$Label <- seurat_obj[[label_col]][, 1]

cat("[INFO] Label distribution after alignment:\n")
print(table(seurat_obj$Label, useNA = "ifany"))


cell_labels_after_QC <- data.frame(
  Cell = colnames(seurat_obj),
  Label = seurat_obj$Label,
  stringsAsFactors = FALSE
)

data.table::fwrite(
  cell_labels_after_QC,
  file.path(out_dir, "cell_labels_after_QC.csv")
)

data.table::fwrite(
  cbind(Cell = rownames(seurat_obj@meta.data), seurat_obj@meta.data),
  file.path(out_dir, "metadata_after_QC.csv")
)

cat("[OK] Labels saved:\n")
cat(file.path(out_dir, "cell_labels_after_QC.csv"), "\n\n")



cat("[INFO] Running NormalizeData with scale.factor = 10000...\n")

seurat_obj <- NormalizeData(
  seurat_obj,
  normalization.method = "LogNormalize",
  scale.factor = scale_factor
)



cat("[INFO] Running FindVariableFeatures, nfeatures = 2000...\n")

seurat_obj <- FindVariableFeatures(
  seurat_obj,
  selection.method = "vst",
  nfeatures = min(n_hvg, nrow(seurat_obj))
)

hvg_genes <- VariableFeatures(seurat_obj)

cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

data.table::fwrite(
  data.frame(HVG = hvg_genes),
  file.path(out_dir, paste0("HVG_", length(hvg_genes), "_gene_list.csv"))
)



cat("[INFO] Running ScaleData on HVGs...\n")

seurat_obj <- ScaleData(
  seurat_obj,
  features = hvg_genes
)



cat("[INFO] Exporting View 1...\n")

raw_hvg_counts_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "counts",
  slot_name = "counts"
)[hvg_genes, ]

normalized_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "data",
  slot_name = "data"
)[hvg_genes, ]

scaled_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "scale.data",
  slot_name = "scale.data"
)[hvg_genes, ]

raw_hvg_cell_by_gene <- t(as.matrix(raw_hvg_counts_gene_by_cell))
normalized_hvg_cell_by_gene <- t(as.matrix(normalized_hvg_gene_by_cell))
scaled_hvg_cell_by_gene <- t(as.matrix(scaled_hvg_gene_by_cell))

raw_hvg_minmax <- minmax_scale_columns(raw_hvg_cell_by_gene)

write_matrix_with_id(
  raw_hvg_cell_by_gene,
  file.path(out_dir, "raw_HVG_counts_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  normalized_hvg_cell_by_gene,
  file.path(out_dir, "normalized_HVG_expr_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  scaled_hvg_cell_by_gene,
  file.path(out_dir, "scaled_HVG_expr_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  raw_hvg_minmax,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 1 saved:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n\n")



cat("[INFO] Preparing scaled HVG matrix for CellEnergy...\n")

scaled_for_cellenergy_path <- file.path(
  out_dir,
  "scaled_HVG_expr_gene_by_cell_for_CellEnergy.csv"
)

write.csv(
  as.data.frame(as.matrix(scaled_hvg_gene_by_cell), check.names = FALSE),
  file = scaled_for_cellenergy_path,
  quote = FALSE,
  row.names = TRUE
)

cat("[INFO] Running CellEnergy::calcGEn...\n")

result <- CellEnergy::calcGEn(
  scaled_for_cellenergy_path,
  verbose = TRUE
)

scEn <- result$scEn
GLNE <- result$GLNE


scEn_df <- as.data.frame(scEn, check.names = FALSE)

if (is.null(rownames(scEn_df)) || any(rownames(scEn_df) == "")) {
  rownames(scEn_df) <- colnames(scaled_hvg_gene_by_cell)
}

scEn_df <- cbind(Cell = rownames(scEn_df), scEn_df)

data.table::fwrite(
  scEn_df,
  file.path(out_dir, "Energy_scEn.csv")
)


GLNE_mat <- as.matrix(GLNE)

write_matrix_with_id(
  GLNE_mat,
  file.path(out_dir, "GLNE_raw_output.csv"),
  id_col = "ID"
)



glne_rows_are_cells <- sum(rownames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))
glne_cols_are_cells <- sum(colnames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))

cat("[INFO] GLNE rows matching cells:\n")
print(glne_rows_are_cells)

cat("[INFO] GLNE cols matching cells:\n")
print(glne_cols_are_cells)

if (glne_rows_are_cells > glne_cols_are_cells) {
  cat("[INFO] GLNE appears to be cells x genes.\n")
  GLNE_cell_by_gene <- GLNE_mat
} else {
  cat("[INFO] GLNE appears to be genes x cells. Transposing.\n")
  GLNE_cell_by_gene <- t(GLNE_mat)
}

common_cells <- intersect(rownames(raw_hvg_cell_by_gene), rownames(GLNE_cell_by_gene))
common_genes <- intersect(colnames(raw_hvg_cell_by_gene), colnames(GLNE_cell_by_gene))

cat("[INFO] Common cells between expression and GLNE:\n")
print(length(common_cells))

cat("[INFO] Common genes between expression and GLNE:\n")
print(length(common_genes))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cells between expression and GLNE.")
}

if (length(common_genes) == 0) {
  stop("[ERROR] No common genes between expression and GLNE.")
}

raw_hvg_cell_by_gene_final <- raw_hvg_cell_by_gene[common_cells, common_genes, drop = FALSE]
GLNE_cell_by_gene <- GLNE_cell_by_gene[common_cells, common_genes, drop = FALSE]


cell_labels_after_QC_final <- cell_labels_after_QC[
  match(common_cells, cell_labels_after_QC$Cell),
  ,
  drop = FALSE
]

data.table::fwrite(
  cell_labels_after_QC_final,
  file.path(out_dir, "cell_labels_after_QC.csv")
)



raw_hvg_minmax_final <- minmax_scale_columns(raw_hvg_cell_by_gene_final)
GLNE_minmax <- minmax_scale_columns(GLNE_cell_by_gene)

write_matrix_with_id(
  raw_hvg_cell_by_gene_final,
  file.path(out_dir, "final_raw_HVG_counts_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  raw_hvg_minmax_final,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_cell_by_gene,
  file.path(out_dir, "final_GLNE_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_minmax,
  file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] Final View 1 saved:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n\n")

cat("[OK] Final View 2 saved:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n\n")



qc_summary_final <- data.frame(
  n_genes_before_qc = nrow(counts),
  n_cells_before_qc = ncol(counts),
  n_genes_after_qc = nrow(seurat_obj),
  n_cells_after_qc = ncol(seurat_obj),
  n_cells_after_glne_alignment = length(common_cells),
  n_genes_after_glne_alignment = length(common_genes),
  min_cells = min_cells,
  min_features = min_features,
  n_hvg = length(hvg_genes),
  scale_factor = scale_factor,
  label_col = label_col
)

data.table::fwrite(
  qc_summary_final,
  file.path(out_dir, "QC_summary.csv")
)

seurat_rds <- file.path(
  out_dir,
  "GSE132188_Seurat_CellEnergy_processed_no_regression.rds"
)

saveRDS(seurat_obj, file = seurat_rds)

cat("\n[DONE] GSE132188 preprocessing finished.\n")
cat("Important outputs:\n")
cat("1. Labels:\n")
cat(file.path(out_dir, "cell_labels_after_QC.csv"), "\n")
cat("2. View 1:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n")
cat("3. View 2:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n")
cat("4. Seurat RDS:\n")
cat(seurat_rds, "\n")
cat("5. QC summary:\n")
cat(file.path(out_dir, "QC_summary.csv"), "\n")

cat("\n[INFO] Final label distribution:\n")
print(table(cell_labels_after_QC_final$Label))

[INFO] Python used by reticulate:
python:         /home/zhanghang/Anaconda/envs/r_cellenergy/bin/python
libpython:      /home/zhanghang/Anaconda/envs/r_cellenergy/lib/libpython3.14.so
pythonhome:     /home/zhanghang/Anaconda/envs/r_cellenergy:/home/zhanghang/Anaconda/envs/r_cellenergy
version:        3.14.5 | packaged by conda-forge | (main, May 20 2026, 00:15:36) [GCC 14.3.0]
numpy:          /home/zhanghang/Anaconda/envs/r_cellenergy/lib/python3.14/site-packages/numpy
numpy_version:  2.4.6

NOTE: Python version was forced by RETICULATE_PYTHON
[INFO] Python module check:
pandas: TRUE 
numpy : TRUE 
scipy : TRUE 

[INFO] Input directory:
/home/zhanghang/GSE132188/mm10 

[INFO] Output directory:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline 

[INFO] Reading matrix.mtx / genes.tsv / barcodes.tsv...
[INFO] Raw matrix dimension, genes x cells:
[1] 27998 11439
[INFO] Seurat object after min.cells / min.features filtering:
An object of class Seurat 
17931 features across 11437 sam

Normalizing layer: counts



[INFO] Running FindVariableFeatures, nfeatures = 2000...


Finding variable features for layer counts



[INFO] Number of HVGs:
[1] 2000
[INFO] Running ScaleData on HVGs...


Centering and scaling data matrix



[INFO] Exporting View 1...
[OK] View 1 saved:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 

[INFO] Preparing scaled HVG matrix for CellEnergy...
[INFO] Running CellEnergy::calcGEn...


2026-06-12 14:35:40.303814: Starting calculation.

2026-06-12 14:36:18.981493: Complete GLNE and cell Energy calculation.



[INFO] GLNE rows matching cells:
[1] 0
[INFO] GLNE cols matching cells:
[1] 2780
[INFO] GLNE appears to be genes x cells. Transposing.
[INFO] Common cells between expression and GLNE:
[1] 2780
[INFO] Common genes between expression and GLNE:
[1] 2000
[OK] Final View 1 saved:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 

[OK] Final View 2 saved:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 


[DONE] GSE132188 preprocessing finished.
Important outputs:
1. Labels:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/cell_labels_after_QC.csv 
2. View 1:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 
3. View 2:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 
4. Seurat RDS:
/home/zhanghang/GSE132188/Seurat_CellEnergy_full_pipeline/GSE132188_Seurat_CellEnergy_processed_no_regressio